# NAM Chain Capture: treinar na nuvem

Este notebook treina a cadeia que você montou no app usando uma GPU grátis do Google. Não precisa mexer em código.

1. Clique em **Ambiente de execução → Executar tudo** (ou aperte **Ctrl+F9**). Se aparecer o aviso de que o notebook não é do Google, clique em **Executar assim mesmo** (Run anyway).
2. Na etapa 2, clique em **Escolher arquivos** e envie o arquivo **.chaincap** que o app salvou na sua pasta Downloads.
3. Espere a barra da etapa 4 chegar ao fim. O arquivo **.nam** baixa sozinho na etapa 5.

Se a etapa 1 avisar que está sem GPU, siga a instrução que aparece e clique em **Executar tudo** de novo.


In [ ]:
#@title 1. Conferir a GPU
import torch
from IPython.display import HTML, display

if not torch.cuda.is_available():
    display(HTML(
        '<div style="font:17px sans-serif;padding:16px;border-radius:10px;background:#5a1f1a;color:#fff;max-width:640px">'
        '<b>Este notebook está sem GPU.</b><br>'
        'Vá em <b>Ambiente de execução → Alterar o tipo de ambiente de execução</b>, escolha <b>GPU T4</b>, '
        'clique em <b>Salvar</b> e depois em <b>Ambiente de execução → Executar tudo</b> de novo.<br>'
        '<small>Se o Colab disser que não há GPU disponível agora, tente mais tarde ou treine no próprio app.</small></div>'))
    raise SystemExit("Sem GPU: siga a instrução acima.")
display(HTML(f'<div style="font:15px sans-serif">GPU pronta: {torch.cuda.get_device_name(0)}</div>'))


In [ ]:
#@title 2. Enviar o pacote da sua cadeia (.chaincap)
import io, json, zipfile
from google.colab import files

display(HTML('<div style="font:15px sans-serif">Clique em <b>Escolher arquivos</b> e escolha o arquivo '
             '<b>.chaincap</b> que o app salvou na sua pasta Downloads.</div>'))
_up = files.upload()
if not _up:
    raise SystemExit("Nenhum arquivo enviado: rode esta etapa de novo.")
_name, _data = next(iter(_up.items()))
try:
    APP_VERSION = json.loads(zipfile.ZipFile(io.BytesIO(_data)).read("settings.json"))["app_version"]
except Exception:
    raise SystemExit("Esse arquivo não é um pacote do NAM Chain Capture (.chaincap). "
                     "No app, clique em \"Treinar grátis na nuvem\" para gerar um.")
open("/content/pacote.chaincap", "wb").write(_data)
print("Pacote recebido:", _name)


In [ ]:
#@title 3. Preparar o treino (cerca de 1 minuto)
import os, subprocess, sys

REPO = "/content/nam-chain-capture"
if not os.path.isdir(REPO):
    # mesma versão do app que gerou o pacote; se não existir, a mais recente
    _r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", f"v{APP_VERSION}",
                         "https://github.com/Longhin08/nam-chain-capture.git", REPO])
    if _r.returncode != 0:
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/Longhin08/nam-chain-capture.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neural-amp-modeler==0.13.0"], check=True)
if REPO + "/poc" not in sys.path:
    sys.path.insert(0, REPO + "/poc")
os.chdir(REPO + "/poc")
from chaincap import colab

JOB = colab.open_package("/content/pacote.chaincap", "/content/cadeia")
V3 = colab.fetch_v3("/content/v3_0_0.wav")
print(f"Pronto para treinar “{JOB['name']}” ({JOB['arch']}, até {JOB['epochs']} épocas).")


In [ ]:
#@title 4. Treinar
REPORT = colab.train(JOB, V3, "/content/resultado", cwd=REPO + "/poc")


In [ ]:
#@title 5. Baixar o resultado
from google.colab import files

RESULT = colab.result_file(JOB, "/content/resultado")
files.download(str(RESULT))
display(HTML(f'<div style="font:15px sans-serif">Baixando <b>{RESULT.name}</b>. Se o navegador perguntar, '
             'permita o download. Depois é só arrastar o arquivo para o app ou mandar para a pedaleira.</div>'))
